In [73]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
import joblib

In [74]:
DATASET_PATH = "../dataset/bccc-cpacket-cloud-ddos-2024-merged.parquet"
df = pd.read_parquet(DATASET_PATH)
df.head()

,src_port,dst_port,duration,packets_count,fwd_packets_count,bwd_packets_count,total_payload_bytes,fwd_total_payload_bytes,bwd_total_payload_bytes,payload_bytes_max,...,max_fwd_payload_bytes_delta_len,mean_fwd_payload_bytes_delta_len,mode_fwd_payload_bytes_delta_len,variance_fwd_payload_bytes_delta_len,std_fwd_payload_bytes_delta_len,median_fwd_payload_bytes_delta_len,skewness_fwd_payload_bytes_delta_len,cov_fwd_payload_bytes_delta_len,label,activity
0,54573,25094,0.000063,3,2,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
1,25094,54573,0.000000,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
2,54573,25094,0.000028,3,1,2,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
3,9147,18060,0.000055,3,2,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
4,18060,9147,0.000000,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign


In [75]:
def preprocess_data(preprocess, df):
    if "activity" in df.columns:
        df = df.drop(columns=["activity"])

    X = df.drop(columns=["label"], errors="ignore")

    if preprocess:
        zero_var_cols = preprocess["zero_var_cols"]
        high_corr_cols = preprocess["high_corr_cols"]
        feature_columns = preprocess["feature_columns"]
    else:
        print("Preprocessing info not found. Please load preprocessing info first.")
        return None, None

    cols_to_drop = list(set(zero_var_cols + high_corr_cols))
    X = df.drop(columns=cols_to_drop, errors="ignore")
    X = X.reindex(columns=feature_columns, fill_value=0).fillna(0)

    return X

In [76]:
MODEL_PATH = "../model/random_forest_model.pkl"
PREPROCESS_INFO_PATH = "../model/random_forest_preprocess_info.pkl"
LABEL_ENCODER_PATH = "../model/label_encoder_info.pkl"

In [77]:
import pickle 

with open(PREPROCESS_INFO_PATH, "rb") as f:
    preprocess_data_info = pickle.load(f)

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)
X = preprocess_data(preprocess_data_info, df)

In [78]:
with open(LABEL_ENCODER_PATH, "rb") as f:
    label_encoder = pickle.load(f)

In [79]:
y_pred = model.predict(X)
y_pred_labels =  label_encoder.inverse_transform(y_pred)
y_true = df["label"]

In [80]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
y_true_encoded = label_encoder.transform(y_true)

# (average='weighted' handles multi-class targets; use 'binary' if you just have 2 classes)
accuracy = accuracy_score(y_true_encoded, y_pred)
precision = precision_score(y_true_encoded, y_pred, average='weighted')
recall = recall_score(y_true_encoded, y_pred, average='weighted')
f1 = f1_score(y_true_encoded, y_pred, average='weighted')

print("--- Classification Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

# b) Calculate ROC-AUC Score
# ROC-AUC requires predicted probabilities rather than class labels
y_prob = model.predict_proba(X)

# multi_class='ovr' (One-vs-Rest) is required for multi-class classification
roc_auc = roc_auc_score(y_true_encoded, y_prob, multi_class='ovr')

print("\n--- ROC-AUC ---")
print(f"ROC-AUC Score: {roc_auc:.4f}")

/home/kenneth/Documents/College/MachineLearning/Project-Machine-Learning/ml_network/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


--- Classification Metrics ---
Accuracy:  0.6460
Precision: 0.4174
Recall:    0.6460
F1-score:  0.5071

--- ROC-AUC ---
ROC-AUC Score: 0.9271


## LogReg
--- Classification Metrics --- <br>
Accuracy:  0.7419 <br>
Precision: 0.9369 <br>
Recall:    0.7419 <br>
F1-score:  0.8145 <br>

--- ROC-AUC --- <br>
ROC-AUC Score: 0.9347 <br>

## KNN
--- Classification Metrics --- <br>
Accuracy:  0.7565 <br>
Precision: 0.7938 <br>
Recall:    0.7565 <br>
F1-score:  0.7575 <br>

--- ROC-AUC ---
ROC-AUC Score: 0.7860 <br>

## XGBOOST
--- Classification Metrics --- <br>
Accuracy:  0.9085 <br>
Precision: 0.9127 <br>
Recall:    0.9085 <br>
F1-score:  0.9049 <br>

--- ROC-AUC ---<br>
ROC-AUC Score: 0.9653 <br>

## Random Forest 
--- Classification Metrics --- <br>
Accuracy:  0.6460<br>
Precision: 0.4174<br>
Recall:    0.6460<br>
F1-score:  0.5071<br>

--- ROC-AUC ---<br>
ROC-AUC Score: 0.9271<br>

## SVM
--- Classification Metrics ---<br>
Accuracy:  0.8962<br>
Precision: 0.9050<br>
Recall:    0.8962<br>
F1-score:  0.8961<br>

--- ROC-AUC ---<br>
ROC-AUC Score: 0.9217<br>


## Log Reg
              precision    recall  f1-score   support

      Attack     0.9787    0.7679    0.8606     17044
      Benign     0.9650    0.7230    0.8266     34918
  Suspicious     0.1151    0.8003    0.2013      2088

    accuracy                         0.7401     54050
   macro avg     0.6863    0.7637    0.6295     54050
weighted avg     0.9365    0.7401    0.8132     54050

KEY METRICS:
  Macro F1:    0.6295
  Weighted F1: 0.8132

## KNN
              precision    recall  f1-score   support

      Attack     0.5849    0.8572    0.6954     34087
      Benign     0.8958    0.7310    0.8050     69836
  Suspicious     0.2550    0.0709    0.1109      4176

    accuracy                         0.7453    108099
   macro avg     0.5786    0.5530    0.5371    108099
weighted avg     0.7730    0.7453    0.7436    108099

KEY METRICS:
  Macro F1:    0.5371  
  Weighted F1: 0.7436  

## SVM
              precision    recall  f1-score   support

      Attack     0.9808    0.7860    0.8726     34087
      Benign     0.9071    0.9866    0.9452     69836
  Suspicious     0.2503    0.2895    0.2685      4176

    accuracy                         0.8964    108099
   macro avg     0.7127    0.6873    0.6954    108099
weighted avg     0.9050    0.8964    0.8961    108099

KEY METRICS:
  Macro F1:    0.6954  ← Primary
  Weighted F1: 0.8961  ← Secondary

## Random Forest
              precision    recall  f1-score   support

      Attack     0.9859    0.8107    0.8898     34087
      Benign     0.9885    0.9144    0.9500     69836
  Suspicious     0.2629    0.9734    0.4140      4176

    accuracy                         0.8840    108099
   macro avg     0.7458    0.8995    0.7513    108099
weighted avg     0.9596    0.8840    0.9103    108099

F1 (macro): 0.7513
F1 (weighted): 0.9103

## XGBoost
              precision    recall  f1-score   support

      Attack     0.9796    0.9098    0.9434     34087
      Benign     0.9929    0.9768    0.9848     69836
  Suspicious     0.5068    0.9387    0.6582      4176

    accuracy                         0.9542    108099
   macro avg     0.8264    0.9418    0.8622    108099
weighted avg     0.9699    0.9542    0.9591    108099

F1 (macro): 0.8622
F1 (weighted): 0.9591